In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import sys

sys.path.append("..")

from component.script import Project, Dataset, JNRBenchmarkModel, rmj


## Set user parameters

In [3]:
project_name = "mtq-refactor"

# Load the project from JSON
project = Project.load(project_name=project_name)


Loaded 1 model(s)
✓ Target set: forest_loss_2020_2024 (static)
✓ Features set: 9 variables
  Static: altitude, protected_area, rivers_dist, roads_dist, slope, subj
  Temporal (year: 2020): forest_gfc, forest_gfc_edge, towns_dist
✓ Target set: forest_loss_2015_2020 (static)
✓ Features set: 9 variables
  Static: altitude, protected_area, rivers_dist, roads_dist, slope, subj
  Temporal (year: 2015): forest_gfc, forest_gfc_edge, towns_dist
Loaded 2 dataset(s)
Project loaded from: /home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/mtq-refactor_project.json
Loaded 23 processed variables


In [4]:
project.list_datasets()


['calibration', 'validation']

In [5]:
calibration_dataset = project.get_dataset("calibration")
calibration_dataset


Dataset(target=LocalRasterVar(name='forest_loss_2015_2020', data_type='raster', year=None, active=True, tags=['deforestation', 'forest_loss', '2015_2020'], path=PosixPath('/home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/data/forest_loss_2015_2020_reprojected_matched.tif'), raster_type='categorical', post_processing=[], processing_history=['reprojected_matched'], default_crs='EPSG:5490', default_resolution=30.0), features=[LocalRasterVar(name='altitude', data_type='raster', year=None, active=True, tags=[], path=PosixPath('/home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/data/altitude_reprojected_matched.tif'), raster_type='continuous', post_processing=[], processing_history=['reprojected_matched'], default_crs='EPSG:5490', default_resolution=30.0), LocalRasterVar(name='forest_gfc', data_type='raster', year=2015, active=True, tags=['forest'], path=PosixPath('/home/jose/workspace/deforisk-nb-with-daniel/deforisk-ju

## Load JNR datasets

JNR datasets differ from the ML model datasets in which variables they need:

| Role | Variable | `fit()` | `apply()` |
|------|----------|---------|-----------|
| Target | deforestation binary (0/1) | ✓ | ✓ |
| Feature | `forest_edge` — distance-to-edge (m) | ✓ | ✓ |
| Feature | `forest` — binary forest at initial year | — | ✓ |
| Feature | `subj` — subjurisdiction IDs | — | ✓ |

One dataset per period with all features works for both `fit()` and `apply()`.
`fit()` only uses the target and `forest_edge_var`; the rest are ignored.

## Instantiate the JNR model

Override `forest_edge_var` / `forest_var` to match this project's variable names.
`defor_threshold` and `max_dist` can be set here as model-wide defaults
or overridden per individual `fit()` call.

In [6]:
jnr = JNRBenchmarkModel(
    name="calibration_jnr",
    forest_edge_var="forest_gfc_edge",  # matches the project variable name
    forest_var="forest_gfc",  # matches the project variable name
    subj_var="subj",  # matches the project variable name (default)
    defor_threshold=99.5,  # distance percentile for the edge threshold
    max_dist=50000,  # maximum distance (m) for the distance-bin arange
    blk_rows=128,
)


## Fit — calibration period

Computes:
1. **`dist_thresh`** — distance-to-edge cutoff (m) from `rmj.dist_edge_threshold`
2. **`dist_bins`** — geometric bin edges from `rmj.compute_dist_bins`

Only the deforestation target and the `forest_edge_var` feature are used.
Results are persisted on the model object; `register()` saves them to project JSON.

In [7]:
jnr.fit(
    dataset=calibration_dataset,
    defor_values=[0],  # pixel value meaning "deforested" in the target raster
    defor_threshold=99.5,  # uncomment to override for this call only
    # max_dist=10000,  # uncomment to override for this call only
    folder=project.folders.rmj_bm,
)

print(f"dist_thresh : {jnr.dist_thresh:.1f} m")
print(f"dist_bins   : {len(jnr.dist_bins)} edges → {len(jnr.dist_bins) - 1} classes")



🔧 JNR fit — period='calibration'
/home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/data/forest_loss_2015_2020_reprojected_matched.tif /home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/data/forest_gfc_reprojected_matched_edge_2015.tif
  dist_thresh=270.0 m
  dist_bins: 30 edges
✓ JNR fit complete — trained_at=2026-03-31T18:33:16.060660
dist_thresh : 270.0 m
dist_bins   : 30 edges → 29 classes


In [8]:
# Register with project — persists dist_thresh, dist_bins, and all metadata
# to the project JSON so the model survives a kernel restart.
jnr.register(project)


  Model registered as project.models['jnr_calibration_jnr']
Project saved to: /home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/mtq-refactor_project.json


## Apply — calibration period

Produces:
1. A **vulnerability map** GeoTIFF via `rmj.vulnerability_map`
2. A **defrate CSV** (deforestation rate per class) via `rmj.defrate_per_class`

The calibration defrate CSV is stored in `jnr.defrate_files["calibration"]`
and passed as `deforate_model` when running the validation period so the
same per-class model rates are applied with a quantity-adjustment correction.

In [9]:
calibration_output = project.folders.rmj_bm / "calibration" / "prob_bm_calibration.tif"

jnr.apply(
    output_file=calibration_output,
    dataset=calibration_dataset,
    time_interval=5,  # 2020 - 2015
    deforate_model=None,  # None for calibration → rates computed from scratch
)



🗺  JNR apply — period='calibration' → prob_bm_calibration.tif
✓ JNR apply complete — /home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/rmj_bm/calibration/prob_bm_calibration.tif


PosixPath('/home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/rmj_bm/calibration/prob_bm_calibration.tif')

In [10]:
import pandas as pd

pd.read_csv(jnr.defrate_files["calibration"]).head(10)


,cat,nfor,ndefor,rate_obs,rate_mod,rate_abs,time_interval,pixel_area,defor_dens
0,1001,5265,5265,1.000000,1.000000,1.000000,5,0.09,0.018000
1,1002,384,384,1.000000,1.000000,1.000000,5,0.09,0.018000
2,1003,7133,7133,1.000000,1.000000,1.000000,5,0.09,0.018000
3,1004,5042,5042,1.000000,1.000000,1.000000,5,0.09,0.018000
4,1005,150,150,1.000000,1.000000,1.000000,5,0.09,0.018000
5,1006,2008,2008,1.000000,1.000000,1.000000,5,0.09,0.018000
6,1007,7984,7984,1.000000,1.000000,1.000000,5,0.09,0.018000
7,1008,8277,8273,0.782811,0.999517,0.999517,5,0.09,0.017991
8,1009,14915,14915,1.000000,1.000000,1.000000,5,0.09,0.018000
9,1010,1565,1565,1.000000,1.000000,1.000000,5,0.09,0.018000


## Apply — validation period

The calibration `dist_bins` are reused (same model instance).
`deforate_model` points to the calibration defrate CSV so that
per-class model rates are applied with a quantity-adjustment correction.

In [11]:
validation_dataset = project.get_dataset("validation")

validation_output = project.folders.rmj_bm / "validation" / "prob_bm_validation.tif"

jnr.apply(
    output_file=validation_output,
    dataset=validation_dataset,
    time_interval=4,  # 2024 - 2020
    deforate_model=jnr.defrate_files.get("calibration"),
)



🗺  JNR apply — period='validation' → prob_bm_validation.tif
✓ JNR apply complete — /home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/rmj_bm/validation/prob_bm_validation.tif


PosixPath('/home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/rmj_bm/validation/prob_bm_validation.tif')

In [12]:
pd.read_csv(jnr.defrate_files["validation"]).head(10)


,cat,nfor,ndefor,rate_obs,nfor_mod,ndefor_mod,rate_obs_mod,rate_mod,rate_abs,time_interval,pixel_area,defor_dens
0,1001,5265,5257,0.802566,5265,5265,1.000000,1.000000,1.000427,4,0.09,0.022510
1,1002,374,374,1.000000,384,384,1.000000,1.000000,1.000427,4,0.09,0.022510
2,1003,7132,7130,0.870594,7133,7133,1.000000,1.000000,1.000427,4,0.09,0.022510
3,1004,5042,5034,0.800418,5042,5042,1.000000,1.000000,1.000427,4,0.09,0.022510
4,1005,150,150,1.000000,150,150,1.000000,1.000000,1.000427,4,0.09,0.022510
5,1006,2007,2007,1.000000,2008,2008,1.000000,1.000000,1.000427,4,0.09,0.022510
6,1007,7921,7921,1.000000,7984,7984,1.000000,1.000000,1.000427,4,0.09,0.022510
7,1008,8042,8042,1.000000,8277,8273,0.782811,0.999517,0.999943,4,0.09,0.022499
8,1009,14902,14894,0.847784,14915,14915,1.000000,1.000000,1.000427,4,0.09,0.022510
9,1010,1556,1547,0.724223,1565,1565,1.000000,1.000000,1.000427,4,0.09,0.022510


## Reload after kernel restart

JNR state (`dist_thresh`, `dist_bins`, `defrate_files`) lives in Pydantic fields —
no pickle file. It reloads automatically from the project JSON.

In [13]:
project2 = Project.load(project_name="mtq-refactor")
jnr_reloaded = project2.models["jnr_calibration_jnr"]
jnr_reloaded.load_model()  # verifies dist_thresh and dist_bins are populated

print(f"dist_thresh : {jnr_reloaded.dist_thresh:.1f} m")
print(f"defrate_files: {dict(jnr_reloaded.defrate_files)}")


Loaded 1 model(s)
✓ Target set: forest_loss_2020_2024 (static)
✓ Features set: 9 variables
  Static: altitude, protected_area, rivers_dist, roads_dist, slope, subj
  Temporal (year: 2020): forest_gfc, forest_gfc_edge, towns_dist
✓ Target set: forest_loss_2015_2020 (static)
✓ Features set: 9 variables
  Static: altitude, protected_area, rivers_dist, roads_dist, slope, subj
  Temporal (year: 2015): forest_gfc, forest_gfc_edge, towns_dist
Loaded 2 dataset(s)
Project loaded from: /home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/mtq-refactor_project.json
Loaded 23 processed variables
  JNR model OK — dist_thresh=270.0 m, 30 bin edges.
dist_thresh : 270.0 m
defrate_files: {}
